# Momentum Investing Research

## What is Momentum Investing?

Momentum is the change in the value of a security over a period, often 12 months, for equities this change in value is represented by the change in the price of its stock as traded on the stockmarket. Momentum investing is where someone chooses stocks with a large increase in value over a certain period and purposfully invests their money in these stocks, with the belief that these equities will continue to increase in value at a rate higher than that of alternative equities. In other words, momentum investing is where investors invest in stocks with high momentum to try and chase higher returns.

## Research Question

This project aims to discover whether momentum investing has shown historically abnormal returns over a prolonged period

## Research Stratergy

This project investigates the performance and resilience of a systematic momentum investment strategy applied to S&P 500 constituents. The strategy ranks stocks based on historical price momentum, constructs a monthly rebalanced portfolio, and evaluates performance after incorporating transaction costs. We will conduct this research over the period of 2009 - 2024.

###  Key Objectives

- Download historical price data
- Inspect data quality
- Calculate basic returns statistics 
- Prepare datasets for portfolio construnction
- Compare portfolio returns against other portfolio returns


In [23]:
# Imported Libraries

import pandas as pd
import numpy as np
from pathlib import Path
import pickle 
import sys

# Data Selection

To account for potential biases I will use historica data of the holding of the S&P 500 as the universe of stocks with which i base this study. By updating the portfolio to only include the holdings within the S&P 500 at a given time period we limit the effects of things like selection bias, whereas if we chose the holdings within the current S&P 500, we are only looking at thise companies which have done successful in their lifecycle up and until this point, which is an impossibility when chossing stocks in the present time. 

## Plan

- Use Kraggle to identify stocks within the S&P 500 at a given time
- Update this for every month start to get the historical holdings and their weightings 
- Clean the data of invalid tickers or replace old tickers 
- Download all ticker price information using yfinance

## Monthly S&P 500 historical holdings 

In [37]:
base_dir = Path.cwd().parent
data_dir = base_dir / "data"
sys.path.append(str(base_dir))

monthly_holdings_df = pd.read_pickle(
    data_dir / "processed" / "monthly_holdings.pkl"
)

with open(data_dir / "processed" / "all_holdings_processed.pkl", "rb") as f:
    all_holdings_processed = pickle.load(f)


print(f"Monthly holdings initial 5 datapoints: \n {monthly_holdings_df.head()} \n")

print(f"First date in the dataframe: {monthly_holdings_df.index.min()}")
print(f"Last date in the dataframe: {monthly_holdings_df.index.max()} \n")
print(f"Total number of holdings: {len(all_holdings_processed)}")

Monthly holdings initial 5 datapoints: 
                                                      Holdings
date                                                         
2009-01-30  [XOM, PG, JNJ, T, CVX, MSFT, GEC, IBM, WMT, PF...
2009-02-27  [XOM, PG, T, JNJ, IBM, MSFT, CVX, WMT, GEC, CS...
2009-03-31  [XOM, T, JNJ, MSFT, PG, CVX, IBM, WMT, GEC, JP...
2009-04-30  [XOM, MSFT, T, PG, JNJ, IBM, GEC, CVX, JPM, CS...
2009-05-29  [XOM, MSFT, JNJ, PG, T, IBM, GEC, JPM, CVX, AA... 

First date in the dataframe: 2009-01-30 00:00:00
Last date in the dataframe: 2024-10-30 00:00:00 

Total number of holdings: 896


## Prices of Holdings Dataframe

In [32]:
with open(data_dir / "processed" / "prices.pkl", "rb") as f:
    prices = pickle.load(f)

print(f"Ticker prices downloaded: {len(prices)} \n")
print(f"First datapoint in downloaded prices: \n\n{list(prices.items())[0]}")

Ticker prices downloaded: 657 

First datapoint in downloaded prices: 

('A', Price            Close        High         Low        Open    Volume
Ticker               A           A           A           A         A
Date                                                                
1999-11-18   26.189781   29.761116   23.808892   27.082613  62546380
1999-11-19   24.032085   25.594542   23.697273   25.557340  15234146
1999-11-22   26.189781   26.189781   23.846093   24.590121   6577870
1999-11-23   23.808884   25.966565   23.808884   25.296939   5975611
1999-11-24   24.441301   24.962121   23.808877   23.883281   4843231
...                ...         ...         ...         ...       ...
2026-08-03  139.800003  140.369995  137.130005  140.289993   1795400
2026-08-04  139.199997  141.000000  136.660004  140.110001   1623700
2026-08-05  141.110001  141.460007  138.550003  140.279999   1266400
2026-08-06  141.339996  141.800003  139.289993  141.580002   1160400
2026-08-07  144.554993  1

## Missing Holdings 

Due to incompatibilities with yfinance and the Kraggle dataset some of the historical holdings of the S&P 500 have to be excluded because no price data can be found of them. This incompatability may just be becasue of formatting inconsistencies, which we have tries to fix in the data_processing.py file, but it is also due to mergers and bankrupties of firms, which has led to their delisting from yfinance. This results in our data being biased as we are only choosing the 'successful' companies, so if the ommitted holdigs have a different momentum to the 'successful' holdings, we may be reducing the returns of our portfolio.  

In [34]:
missing_tickers = [
    ticker
    for ticker in all_holdings_processed
    if ticker not in prices
]

print(f"First 10 missing ticker: \n\n{missing_tickers[:10]} \n")
print(f"Quantity of missing ticker: {len(missing_tickers)}")

First 10 missing ticker: 

['0R01', '3EC', '4XS', '6COP', '8686', 'A60', 'AABA', 'ABC', 'ABMD', 'ACAS'] 

Quantity of missing ticker: 239


In [ ]:
from src.backtest import(
    monthly_portfolio_return,
    periodic_momentum_returns
)

from src.portfolio import normalised_weightings

net_return, turnover, new_weights = monthly_portfolio_return(
        "2023-06-20",
        "2024-06-28",
        normalised_weightings("2023-06-20", prices)["Weights"]
    )

print(f"Net Returns: {net_return}")
print(f"Turnover: {turnover}")
print(f"Net Weights: {new_weights}")

total_return, total_turnover = periodic_momentum_returns("2024-03-28", "2024-06-28")

print(f"Total Return: {total_return}")
print(f"Total Turnover: {total_turnover}")


Net Returns: 0.44479668442238013
Turnover: 0.83567695
Net Weights: VST     0.065065
SMCI    0.055691
NVDA    0.050312
CEG     0.046530
FSLR    0.043461
NRG     0.042622
GE      0.040086
DECK    0.039387
HWM     0.038992
QCOM    0.038627
MRNA    0.038468
CRWD    0.038254
MU      0.038234
WDC     0.037356
GDDY    0.036227
LDOS    0.035385
ETN     0.035306
ANET    0.035270
APH     0.035233
TER     0.035173
TT      0.035085
KLAC    0.034959
WAB     0.034886
LLY     0.034723
NTAP    0.034668
Name: Weights, dtype: float64
Total Return: -0.009963183267938658
Total Turnover: 1.2937675499999999
